# Goal

Серия экспериментов. Треним за 5 поколений по 6 млн шагов с последующим отбором чемпионов. Наследники этих чемпионов будут использоваться на следующем шаге.

Данный эксперимент - это ЧЕТВЁРТЫЙ шаг. Здесь оттачиваем навыки агента #132.

Треним агента на полноцветном материале, т.е. `observation_space_shape=(3, 84, 84)`. Также используется агент с переделанным CNN слоем. По прошлым экспериментам видно (см. grad_norm), что CNN "задыхается" и его постоянно колбасит при смене палитры уровней. Переделка = убран лишний ReLU + доп. Linear слой + LayerNorm на выходе. Также после трансформера добавлен доп. LayerNorm, т.к. это типа must-have, когда в трансформере используется pre_ln.

# set_hyperparameters

In [1]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    generations_count = 5
    generation_ind = 3
    generation_steps_count = 6_000_000
    all_generations_steps_count = generation_steps_count * generations_count
    learn_rate_range = (0.00025, 0.00025 * 0.1)
    ent_coef_range = (0.05, 0.05 * 0.1)
    tau_range = (0.5, 0.1)
    ####
    
    import random
    HP.system.random_seed = random.randint(1, 100)
    HP.system.is_torch_deterministic = True
    HP.system.is_torch_compile = True
    
    HP.env.ident = 'FrostbiteNoFrameskip-v4'
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6
    HP.env.idle_penalty = 0
    HP.env.life_lost_penalty = 0

    HP.agent.parent = '17e_ppo_tr_atari_mp_15:132'
    HP.agent.observation_space_shape = (3, 84, 84) 
    HP.agent.layers_count = 3 # number of transformer layers
    HP.agent.heads_count = 4 # number of heads used in multi-head attention
    HP.agent.d_model = 256 # dimension of the transformer
    HP.agent.cnn_expand_dim = 1024 # dimension of the transformer
    HP.agent.obs_sequence_length = 4 # length observation chain agent incepts
    HP.agent.action_plan_length = 10 # number of actions agent must think upfront about
    HP.agent.positional_encoding = 'learned' # positional encoding type of the transformer: "", "absolute", "learned"
    
    # Video params
    HP.video.capture_policy = 'every(500000)' # video capture policy depending on steps
    HP.video.capture_env_rams = None
    HP.video.capture_env_ram_patches = None
    HP.video.break_on_level_passed = False
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = generation_steps_count # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.rollout_env_rams = [
        'com.develorium.neurolab.frostbite_ram:level1_101:1', 
        'com.develorium.neurolab.frostbite_ram:level4_101:1', # bear level, day
        'com.develorium.neurolab.frostbite_ram:level5_101:1', # bear level, night
    ]
    HP.ppo.rollout_env_ram_patches = [
        ['no_score', 'three_lives', 'no_igloo'],
    ]

    tau_change_speed = (tau_range[1] - tau_range[0]) / generations_count
    tau_a = tau_range[0] + tau_change_speed * generation_ind
    tau_b = tau_a + tau_change_speed
    HP.ppo.tau = f'linear({tau_a}, {tau_b})' # temperature to inject randomness during actions selection (Gumbel Max)
    # HP.ppo.tau = 'linear(0.5, 0.34)' # temperature to inject randomness during actions selection (Gumbel Max)
    
    HP.ppo.epochs_count = 2 
    HP.ppo.minibatches_count = 16 # obs shape (3, 84, 84) -> 16, obs shape (1, 84, 84) -> 8
    learn_rate_change_speed = (learn_rate_range[1] - learn_rate_range[0]) / generations_count
    learn_rate_a = learn_rate_range[0] + learn_rate_change_speed * generation_ind
    learn_rate_b = learn_rate_a + learn_rate_change_speed
    HP.ppo.learn_rate = f'linear({learn_rate_a}, {learn_rate_b})'
    # HP.ppo.learn_rate = 'linear(0.00025, 0.00015)'
    HP.ppo.optimizer = 'AdamW'
    
    HP.ppo.vf_coef = 0.5 # coefficient of the value function within loss function
    ent_coef_change_speed = (ent_coef_range[1] - ent_coef_range[0]) / generations_count
    ent_coef_a = ent_coef_range[0] + ent_coef_change_speed * generation_ind
    ent_coef_b = ent_coef_a + ent_coef_change_speed
    HP.ppo.ent_coef = f'linear({ent_coef_a}, {ent_coef_b})' # coefficient of the entropy member within loss function
    # HP.ppo.ent_coef = 'linear(0.05, 0.03)' # coefficient of the entropy member within loss function
    HP.ppo.consistency_coef = 0.1
    HP.ppo.prediction_coef = 0.1
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # the target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    return HP
# @launchit.stop

# Results
<TBD>

Порядочное кол-во агентов научилось проходить 4-е уровня, но 5-ый оказывается не по зубам. Пытался выяснить в чём дело с помощью `17e_explore_01.ipynb` - заставлял играть на 5-ом уровне с 8-ью жизнями. Даже 8 жизней не достаточно, чтобы агент прошёл 5-ый уровень, хотя иглу собирает, и даже вроде пытается двигаться в направлении, но всегда гибнет по какой-то глупости. Также заметил, что агент очень детерминирован - каждый раз играет по одному и тому же сценарию. Первую жизнь, например, тратит одинаково. После доп. ковыряния через `17e_explore_01.ipynb` выяснилось, что логиты действий после tau-скейлинга практически превратились в one-hot encoding. Значит, агент настолько одеревенел, что практически не поддаётся обучению. Вообще подозреваю, что агент выучился механически проходить уровни, не обращая внимания на обстоятельства?

Возможно, дело в слишком заумной системе с Гумбель-максингом: вместо одного параметра, который бы управлял исследовательским фактором агента - ent_coef, появился ещё один - tau. В итоге система оказывается под давлением сразу с двух сторон: уменьшающийся ent_coef двигает логиты к one-hot encoding, так ещё и уменьшающийся tau также очень сильно давит в этом направлении. В итоге скорость одеревенения получается завышенной (м.б. квадратично даже?).

Как следствие, это может объяснять и высокий градиент CNN - системе просто не остаётся ничего делать, как толкать градиент взад по графу в надежде на то, что тем самым измеятся логиты.

Ну если на сам по себе отжиг tau роялит, то скорее скорость отжига. Tau так быстро отжигается, что система просто не успевает подстраиваться и в итоге выдаёт высокие градиенты.

Короче, последний шаг в серии 22 не имеют смысла, т.к. агент уже ничему не научится из-за вырожденных логитов.

<img src="./img/levels_passed.png">
<img src="./img/reward.png">
<img src="./img/grad_norms_slice.png">

**Выводы**
1) проблема с обучаемостью агента лежит в tau (сам tau или скорость отжига, или и то и другое)